In [1]:
import os
import random
import torch
import numpy as np
import matplotlib.pyplot as plt

from transformers import AutoTokenizer, AutoModel
from datasets import load_dataset
from tqdm import tqdm
from sklearn.decomposition import PCA
from sklearn.metrics.pairwise import cosine_similarity

###############################################################################
# 1) Define Arguments
###############################################################################
args = {
    # Two different models
    "models": ["nlpaueb/sec-bert-base", "bert-base-uncased"],
    # Three different datasets
    "datasets": [
        {
            "name": "yelp_review_full",  # Food reviews
            "config": None,
            "split": "train",
            "text_column": "text",
        },
        {
            "name": "wikitext",  # Wiki comments/text
            "config": "wikitext-2-raw-v1",
            "split": "train",
            "text_column": "text",
        },
        {
            "name": "ag_news",  # News
            "config": None,
            "split": "train",
            "text_column": "text",
        },
    ],
    "max_texts": 5000,  # Maximum texts to load per dataset for speed
    "batch_size": 64,
    "drift_threshold_std": 3.0,
    "window_size": 50,
    "variance_threshold": 1e-4,
    # For artificial drift generation
    "num_drift_levels": 5,  # How many steps of "gradual" drift to simulate
    "drift_strengths": [0.0, 0.25, 0.5, 0.75, 1.0],  # fraction of words to shuffle
    "pca_components": 30,  # Number of PCA components
    "output_dir": "results_multidataset",
}

os.makedirs(args["output_dir"], exist_ok=True)

###############################################################################
# 2) Utility Functions
###############################################################################


def batch_generator(data, batch_size=32):
    """Generate batches from a list of data."""
    for i in range(0, len(data), batch_size):
        yield data[i : i + batch_size]


def extract_cls_embeddings(model, tokenizer, texts, device):
    """
    Tokenize and extract [CLS] embeddings from a batch of texts
    using a generic AutoModel (base transformer).
    """
    encodings = tokenizer(
        texts, return_tensors="pt", padding=True, truncation=True, max_length=128
    )
    input_ids = encodings["input_ids"].to(device)
    attention_mask = encodings["attention_mask"].to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        # last_hidden_state shape: [batch_size, seq_len, hidden_dim]
        # We'll use the [CLS] token (index 0) as the embedding
        cls_embeddings = outputs.last_hidden_state[:, 0, :]
    return cls_embeddings.cpu().numpy()


def cluster_drift_indices(
    drift_indices,
    distance_threshold=128,
    ignore_solo_outliers=False,
    min_cluster_size=2,
):
    """
    Post-process a list of integer drift indices by grouping those
    that are within `distance_threshold` of each other.
    """
    if not drift_indices:
        return []

    drift_indices = sorted(drift_indices)
    clusters = []
    current_cluster = [drift_indices[0]]

    for i in range(1, len(drift_indices)):
        if drift_indices[i] - drift_indices[i - 1] <= distance_threshold:
            current_cluster.append(drift_indices[i])
        else:
            clusters.append(current_cluster)
            current_cluster = [drift_indices[i]]
    clusters.append(current_cluster)

    if ignore_solo_outliers:
        clusters = [c for c in clusters if len(c) >= min_cluster_size]
    return clusters


def choose_cluster_representatives(clusters, mode="first"):
    """
    Pick a single integer "representative" from each cluster of drift indices.
    mode can be 'first', 'last', 'mean', or 'median'.
    """
    representatives = []
    for cluster in clusters:
        if mode == "first":
            representatives.append(cluster[0])
        elif mode == "last":
            representatives.append(cluster[-1])
        elif mode == "mean":
            representatives.append(int(sum(cluster) / len(cluster)))
        elif mode == "median":
            mid = len(cluster) // 2
            sorted_cluster = sorted(cluster)
            if len(cluster) % 2 == 1:
                representatives.append(sorted_cluster[mid])
            else:
                lower = sorted_cluster[mid - 1]
                upper = sorted_cluster[mid]
                representatives.append(int((lower + upper) / 2))
        else:
            raise ValueError(f"Unknown mode: {mode}")
    return representatives


def introduce_gradual_drift(text_list, fraction_shuffle=0.5):
    """
    For demonstration, we'll "shuffle" a fraction of the words in each text
    as a simple artificial drift. fraction_shuffle=0 => no drift, 1.0 => fully shuffled.
    """
    new_texts = []
    for txt in text_list:
        words = txt.split()
        if len(words) < 2:
            # Too short, skip shuffling
            new_texts.append(txt)
            continue

        # Number of words to shuffle
        k = int(len(words) * fraction_shuffle)
        if k < 1:
            new_texts.append(txt)
            continue

        # We'll just shuffle k words from the text
        indices = list(range(len(words)))
        random.shuffle(indices)
        shuffle_indices = indices[:k]

        # Extract those words
        to_shuffle = [words[i] for i in shuffle_indices]
        random.shuffle(to_shuffle)

        # Put them back
        for i, idx in enumerate(shuffle_indices):
            words[idx] = to_shuffle[i]

        new_texts.append(" ".join(words))
    return new_texts


###############################################################################
# 3) DriftDetector Class
###############################################################################
class DriftDetector:
    """
    Simple drift detector that:
    1) Initializes a prototype embedding vector from a baseline set.
    2) Batches through data, computes average [CLS] embedding for each batch.
    3) Compares new batch embedding to the prototype with cosine similarity.
    4) If similarity < threshold (or the variance is too low), mark drift.
    5) Dynamically update the prototype after each batch.
    """

    def __init__(
        self, model, tokenizer, device, batch_generator, args, pca_transform=None
    ):
        self.model = model
        self.tokenizer = tokenizer
        self.device = device
        self.batch_generator = batch_generator
        self.args = args

        # Optional PCA transform
        self.pca_transform = pca_transform

        self.prototype = None
        self.prototypes = []
        self.drifts = []
        self.cosine_scores = []

    def initialize_baseline(self, texts):
        """Compute an initial prototype from the first half of the data (baseline)."""
        embeddings = []
        for batch in self.batch_generator(texts, self.args["batch_size"]):
            cls_emb = extract_cls_embeddings(
                self.model, self.tokenizer, batch, self.device
            )
            embeddings.append(cls_emb)
        all_embeddings = np.concatenate(embeddings, axis=0)

        # Optionally apply PCA
        if self.pca_transform is not None:
            all_embeddings = self.pca_transform.transform(all_embeddings)

        self.prototype = np.mean(all_embeddings, axis=0)
        self.prototypes.append(self.prototype)

    def detect_drifts(self, texts):
        """Go through the data in batches to detect potential drifts."""
        batch_index = 0
        for batch in tqdm(self.batch_generator(texts, self.args["batch_size"])):
            batch_embeddings = extract_cls_embeddings(
                self.model, self.tokenizer, batch, self.device
            )

            # Apply PCA if needed
            if self.pca_transform is not None:
                batch_embeddings = self.pca_transform.transform(batch_embeddings)

            # Mean embedding for current batch
            mean_emb = batch_embeddings.mean(axis=0, keepdims=True)
            sim = cosine_similarity(mean_emb, [self.prototype])[0][0]
            self.cosine_scores.append(sim)

            # If we have enough scores, check for drift
            if len(self.cosine_scores) >= self.args["window_size"]:
                threshold = self._calculate_threshold()
                variance = self._calculate_variance()
                if sim < threshold or (
                    variance is not None and variance < self.args["variance_threshold"]
                ):
                    # The drift index is the data index, not the batch number
                    drift_index = batch_index * self.args["batch_size"]
                    self.drifts.append(drift_index)

            # Update the prototype
            self._update_prototype(batch_embeddings)
            batch_index += 1

    def _calculate_threshold(self):
        """Use rolling mean and std to define a drift threshold."""
        recent_scores = self.cosine_scores[-self.args["window_size"] :]
        mean_val = np.mean(recent_scores)
        std_val = np.std(recent_scores)
        threshold = mean_val - (self.args["drift_threshold_std"] * std_val)
        return threshold

    def _calculate_variance(self):
        """Variance of the most recent cosine similarity window."""
        if len(self.cosine_scores) < self.args["window_size"]:
            return None
        recent_scores = self.cosine_scores[-self.args["window_size"] :]
        return np.var(recent_scores)

    def _update_prototype(self, batch_embeddings):
        """Update the prototype by combining current prototype with new embeddings."""
        delta = batch_embeddings - self.prototype
        distances = np.linalg.norm(delta, axis=1)
        weights = np.exp(-distances / 2.0)
        weighted_sum = np.sum(weights[:, None] * delta, axis=0)
        self.prototype += weighted_sum / np.sum(weights)
        self.prototypes.append(self.prototype)


###############################################################################
# 4) Main Flow
###############################################################################
def main():
    # Decide on device
    if torch.backends.mps.is_available():
        device = torch.device("mps")
    elif torch.cuda.is_available():
        device = torch.device("cuda")
    else:
        device = torch.device("cpu")
    print("Using device:", device)

    # Create output dir
    os.makedirs(args["output_dir"], exist_ok=True)

    # We'll store drift results (cosine scores) for each (dataset, model, drift_level, pca/no-pca)
    results = []

    for dataset_info in args["datasets"]:
        dataset_name = dataset_info["name"]
        dataset_config = dataset_info["config"]
        dataset_split = dataset_info["split"]
        text_col = dataset_info["text_column"]

        print(f"\n=== Loading dataset: {dataset_name} ===")
        # Load dataset
        ds = load_dataset(dataset_name, dataset_config, split=dataset_split)
        texts = ds[text_col]

        # Possibly shuffle and slice
        texts = list(texts)
        random.shuffle(texts)
        if args["max_texts"] > 0 and len(texts) > args["max_texts"]:
            texts = texts[: args["max_texts"]]

        # We will split the dataset in half:
        half_point = len(texts) // 2
        baseline_texts = texts[:half_point]

        for model_name in args["models"]:
            print(f"\n--- Using Model: {model_name} ---")

            # Load model & tokenizer
            tokenizer = AutoTokenizer.from_pretrained(model_name)
            # Use a plain AutoModel for embeddings (not a classification head)
            model = AutoModel.from_pretrained(model_name)
            model.to(device)
            model.eval()

            # -------------------------------------------------------
            # We'll iterate through different drift strengths
            # 0.0 => no drift, up to 1.0 => fully shuffled
            # -------------------------------------------------------
            for drift_strength in args["drift_strengths"]:
                print(f"\nSimulating drift with strength={drift_strength}")

                # We artificially drift the *second half* of the data
                drifted_texts = introduce_gradual_drift(
                    texts[half_point:], fraction_shuffle=drift_strength
                )
                # Combine baseline + drifted
                test_texts = baseline_texts + drifted_texts

                # 1) No PCA scenario
                detector_no_pca = DriftDetector(
                    model=model,
                    tokenizer=tokenizer,
                    device=device,
                    batch_generator=batch_generator,
                    args=args,
                    pca_transform=None,
                )
                # Initialize baseline on the first half
                detector_no_pca.initialize_baseline(baseline_texts)
                detector_no_pca.detect_drifts(test_texts)  # entire set

                # 2) PCA scenario
                # First, fit PCA on baseline embeddings
                baseline_embs = []
                for b in batch_generator(baseline_texts, args["batch_size"]):
                    emb_b = extract_cls_embeddings(model, tokenizer, b, device)
                    baseline_embs.append(emb_b)
                baseline_embs = np.concatenate(baseline_embs, axis=0)

                pca = PCA(n_components=args["pca_components"])
                pca.fit(baseline_embs)

                detector_pca = DriftDetector(
                    model=model,
                    tokenizer=tokenizer,
                    device=device,
                    batch_generator=batch_generator,
                    args=args,
                    pca_transform=pca,
                )
                # Initialize baseline (with PCA)
                detector_pca.initialize_baseline(baseline_texts)
                detector_pca.detect_drifts(test_texts)

                # Store results
                results.append(
                    {
                        "dataset": dataset_name,
                        "model": model_name,
                        "drift_strength": drift_strength,
                        "pca": False,
                        "cosine_scores": detector_no_pca.cosine_scores,
                    }
                )
                results.append(
                    {
                        "dataset": dataset_name,
                        "model": model_name,
                        "drift_strength": drift_strength,
                        "pca": True,
                        "cosine_scores": detector_pca.cosine_scores,
                    }
                )

                # Just a small printout: number of drift detections found
                print(f"  Detected Drifts (no PCA): {len(detector_no_pca.drifts)}")
                print(f"  Detected Drifts (PCA): {len(detector_pca.drifts)}")

    # -------------------------------------------------------------------------
    # 5) Plot or Analyze Results
    # For demonstration, we'll produce simple plots of "cosine_scores" vs. index
    # for each drift strength, comparing PCA vs. no-PCA.
    # -------------------------------------------------------------------------
    print("\n=== Plotting results for each (dataset, model, drift_strength) ===")

    # Group results by (dataset, model, drift_strength)
    from collections import defaultdict

    grouped = defaultdict(list)

    for r in results:
        key = (r["dataset"], r["model"], r["drift_strength"])
        grouped[key].append(r)

    for (dataset_name, model_name, drift_strength), group_vals in grouped.items():
        # group_vals has 2 dicts: one for PCA=False, one for PCA=True
        plt.figure(figsize=(10, 6))
        for gv in group_vals:
            label_suffix = "PCA" if gv["pca"] else "No PCA"
            plt.plot(gv["cosine_scores"], label=f"{label_suffix}")

        plt.title(f"{dataset_name} | {model_name} | drift={drift_strength}")
        plt.xlabel("Batch Index")
        plt.ylabel("Cosine Similarity")
        plt.legend()
        plt.grid(True, linestyle="--", alpha=0.5)

        fname = f"{dataset_name}_{model_name}_drift{drift_strength}.png"
        save_path = os.path.join(args["output_dir"], fname)
        plt.savefig(save_path)
        plt.close()
        print(f"  Saved plot: {save_path}")

    print("\nAll done!")


# Execute main
if __name__ == "__main__":
    main()

/Users/jasper.bruin/miniconda3/envs/driftwatch/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: mps

=== Loading dataset: yelp_review_full ===


Generating test split: 100%|██████████| 50000/50000 [00:00<00:00, 2113276.30 examples/s]



--- Using Model: nlpaueb/sec-bert-base ---

Simulating drift with strength=0.0


79it [00:30,  2.57it/s]
79it [00:30,  2.56it/s]


...

  Detected Drifts (no PCA): 28
  Detected Drifts (PCA): 0

Simulating drift with strength=1.0


79it [00:30,  2.58it/s]
79it [00:31,  2.52it/s]


  Detected Drifts (no PCA): 28
  Detected Drifts (PCA): 0

--- Using Model: bert-base-uncased ---

Simulating drift with strength=0.0


79it [00:31,  2.54it/s]


KeyboardInterrupt: 